# S06 — ICD chapter metadata
Owner-run CUDA semantic/combined annotation over existing prepared documents. No LLM generation, no chunk rebuild, no QA catalog. Clinical audit/calibration is deferred. See `docs/ICD_STAGE6.md`. One writer per backup directory; a single GPU session is sufficient. All outputs remain provisional and ICD boosts/filters stay disabled.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, shutil, subprocess, zipfile
BUNDLE = Path('/content/drive/MyDrive/ViBioMIR/stage6_icd_colab.zip')
PROJECT = Path('/content/vibiomir_stage6')
MODE = 'combined'  # emits rules, semantic and combined in one pass
RUN_DIR = Path('/content/icd_stage6_' + MODE)
BACKUP_DIR = Path('/content/drive/MyDrive/ViBioMIR/icd_stage6_' + MODE)
ENCODER_BATCH_SIZE = 16  # T4/L4 start; try 64 on A100 after the pilot
PADDED_TOKENS = 8192  # A100 pilot: 32768
DOCUMENTS_PER_WAVE = 16  # A100 pilot: 32; bounded to 128
STOP_AFTER_BATCHES = 1  # then None; rerun the run cell to continue
assert MODE in ('semantic', 'combined')
assert BUNDLE.is_file(), 'Upload the S06 bundle and edit BUNDLE first.'
assert RUN_DIR.resolve() != BACKUP_DIR.resolve()
os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false', OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1')


In [ ]:
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    required = sum(info.file_size for info in archive.infolist())
    assert shutil.disk_usage('/content').free > required * 2 + 4 * 1024**3, 'Free local disk for inputs, model cache and outputs.'
    for info in archive.infolist():
        target = (PROJECT / info.filename).resolve()
        assert PROJECT.resolve() in target.parents, 'Unsafe ZIP path'
        assert (info.external_attr >> 16) & 0o170000 != 0o120000, 'Symlink ZIP entries are not allowed'
    archive.extractall(PROJECT)
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-stage6-colab.txt'], check=True)
print('If Colab requests a runtime restart, restart and rerun setup before proceeding.')


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a CUDA GPU runtime; TPU is not supported.'
print({'gpu': torch.cuda.get_device_name(0), 'input': json.loads((PROJECT / 'inputs/manifest.json').read_text())['documents']})
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_stage6*.py', '-v'], check=True)


In [ ]:
command = [sys.executable, '-m', 'vibiomir.icd_pipeline', 'build',
    '--input-dir', str(PROJECT / 'inputs'), '--output-dir', str(RUN_DIR),
    '--config', str(PROJECT / 'configs/icd_stage6.json'), '--mode', MODE,
    '--tokenizer-dir', str(PROJECT / 'tokenizer'), '--backup-dir', str(BACKUP_DIR),
    '--encoder-batch-size', str(ENCODER_BATCH_SIZE), '--padded-tokens', str(PADDED_TOKENS),
    '--documents-per-wave', str(DOCUMENTS_PER_WAVE)]
if STOP_AFTER_BATCHES is not None:
    command += ['--stop-after-batches', str(STOP_AFTER_BATCHES)]
subprocess.run(command, check=True)
state = json.loads((RUN_DIR / 'manifest.json').read_text())
print({k: state.get(k) for k in ('state','documents','annotation_records','calibration')})
if state['state'] != 'complete':
    print('Pilot paused safely. Set STOP_AFTER_BATCHES=None and rerun this cell. Committed batches are reused.')


In [ ]:
state = json.loads((RUN_DIR / 'manifest.json').read_text())
if state['state'] == 'complete':
    subprocess.run([sys.executable, '-m', 'vibiomir.icd_pipeline', 'validate',
        '--input-dir', str(PROJECT / 'inputs'), '--output-dir', str(RUN_DIR),
        '--config', str(PROJECT / 'configs/icd_stage6.json')], check=True)
    from vibiomir.chunk_colab import save_checkpoint
    save_checkpoint(RUN_DIR, BACKUP_DIR)
    shutil.copy2(RUN_DIR / 'MODEL_REGISTRY.json', BACKUP_DIR / 'MODEL_REGISTRY.json')
    print('Download the whole backup directory, including VALIDATION.json, MODEL_REGISTRY.json and every batch:', BACKUP_DIR)
else:
    print('Finish the remaining batches before full technical validation.')
